In [ ]:
from data_loaders.cars_data import load_train_val_test
from utils.preprocessing import get_feature_types, build_preprocessor


X_train, y_train, X_val, y_val, X_test = load_train_val_test(
    "../data/train.csv",
    "../data/test.csv",
)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape)


numeric_features, categorical_features = get_feature_types(X_train)
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

preprocessor = build_preprocessor(numeric_features, categorical_features)


In [1]:
from data_loaders.cars_data import load_full_train_and_test
from utils.feature_engineering import add_model_engine_rarity_cv
from wrappers.baseline_nn_pipeline import build_flexible_nn_pipeline
from utils.feature_selection import candidate_selectors

from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, r2_score, make_scorer
import numpy as np

# 1) Load full labeled train + test (no split, no imputation)
X_full, y_full, X_test = load_full_train_and_test(
    train_path="../data/train.csv",
    test_path="../data/test.csv",
    mapping_dir="../mapping",
)

# 2) Add rarity feature using X_full
X_full, X_test = add_model_engine_rarity_cv(
    X_full,
    X_test,
    model_col="model",
    engine_col="engineSize",
    new_col="model_engine_freq",
    log_scale=True,
)

# 3) Build base pipeline: [preprocess] -> [feature_sel] -> [model]
base_pipe = build_flexible_nn_pipeline(X_full, random_state=42)

# 4) Wrap with log-target
log_pipe = TransformedTargetRegressor(
    regressor=base_pipe,
    func=np.log1p,
    inverse_func=np.expm1,
)

# 5) RMSE scorer
def rmse_func(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return np.sqrt(mse)

rmse_scorer = make_scorer(rmse_func, greater_is_better=False)

# 6) KFold (K=5)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 7) FS candidates
selectors = candidate_selectors(random_state=42)

# 8) Param grid over FS + NN
param_grid = [
    {
        "regressor__feature_sel": list(selectors.values()),
        "regressor__model__hidden_layer_sizes": [(64, 32), (128, 64)],
        "regressor__model__learning_rate_init": [1e-3, 5e-4],
        "regressor__model__alpha": [1e-4, 1e-3],
    }
]

search = GridSearchCV(
    estimator=log_pipe,
    param_grid=param_grid,
    scoring=rmse_scorer,
    cv=kf,           # <-- 5-fold CV, each fold does its own imputer/FS/model
    n_jobs=-1,
    verbose=2,
)

search.fit(X_full, y_full)

print("Best params:", search.best_params_)
print("Best CV RMSE (negative):", search.best_score_)


[TRAIN] duplicate carID count: 0
[TEST] duplicate carID count: 0
Fitting 5 folds for each of 48 candidates, totalling 240 fits


/Users/karaca/src/MachineLearningProject-NOVAIMS2025/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/karaca/src/MachineLearningProject-NOVAIMS2025/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/karaca/src/MachineLearningProject-NOVAIMS2025/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/karaca/src/MachineLearningProject-NOVAIMS2025/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:788: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
/Users/karaca/src/MachineLearningProject-NOVAIMS2025/.venv/lib/p

[CV] END regressor__feature_sel=passthrough, regressor__model__alpha=0.0001, regressor__model__hidden_layer_sizes=(64, 32), regressor__model__learning_rate_init=0.001; total time=   9.2s
[CV] END regressor__feature_sel=passthrough, regressor__model__alpha=0.0001, regressor__model__hidden_layer_sizes=(64, 32), regressor__model__learning_rate_init=0.0005; total time=   9.1s
[CV] END regressor__feature_sel=passthrough, regressor__model__alpha=0.0001, regressor__model__hidden_layer_sizes=(64, 32), regressor__model__learning_rate_init=0.001; total time=   9.2s
[CV] END regressor__feature_sel=passthrough, regressor__model__alpha=0.0001, regressor__model__hidden_layer_sizes=(64, 32), regressor__model__learning_rate_init=0.0005; total time=   9.2s
[CV] END regressor__feature_sel=passthrough, regressor__model__alpha=0.0001, regressor__model__hidden_layer_sizes=(64, 32), regressor__model__learning_rate_init=0.001; total time=   9.2s
[CV] END regressor__feature_sel=passthrough, regressor__model__

KeyboardInterrupt: 

In [ ]:
import numpy as np

# Make a copy so we don't mess with X_val
val_results = X_val.copy()

# Add true and predicted prices
val_results["price_true"] = y_val  # works if indices align (they should)
val_results["price_pred"] = y_pred_val

# Add error columns
val_results["error"] = val_results["price_pred"] - val_results["price_true"]
val_results["abs_error"] = np.abs(val_results["error"])
val_results["rel_error_pct"] = (
    val_results["abs_error"] / val_results["price_true"]
) * 100  # percentage error

# Sort by highest absolute error
val_results_sorted = val_results.sort_values("abs_error", ascending=False)


In [ ]:

cols_to_show = [
    "Brand", "model", "year", "mileage",
    "fuelType", "transmission","engineSize",
    "price_true", "price_pred", "abs_error", "rel_error_pct",
]

val_results_sorted[cols_to_show].head(20)

